# Смотрелка данных

Показывает одну куку целиком: её события по порядку с паузами между ними и ключевые признаки
рядом с медианами по ботам и людям.

Нужна, чтобы проверять гипотезы глазами до того, как считать метрики.

In [1]:
import sys, warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.width", 220)
pd.set_option("display.max_rows", 200)

import avito_lib as L

train, test, events = L.load_data()
lab = train.set_index("cookie_id")["target"]
EV = L.clip_to_window(events, train)
EV["dt"] = EV.groupby("cookie_id")["event_ts"].diff().dt.total_seconds()

pop = L.build_population_stats(events, train, test)
X, GROUPS = L.build_features(train, events, pop)
X.index = train["cookie_id"].values
MED = X.assign(t=lab.reindex(X.index).values).groupby("t").median()
print("готово:", len(train), "кук,", len(EV), "событий в окнах")

готово: 11091 кук, 198436 событий в окнах


In [2]:
KEY = ["n_events", "dt_median", "dt_cv", "ptr_frac", "max_search_page",
       "n_cat_uniq", "n_loc_uniq", "item_pop_mean", "photo_per_view", "q_repeat_frac"]


def show(cookie_id=None, target=None, n=40, seed=None):
    """Показать одну куку. Без аргументов — случайная. target=1 — случайный бот."""
    if cookie_id is None:
        pool = lab.index if target is None else lab[lab == target].index
        cookie_id = pd.Series(pool).sample(1, random_state=seed).iat[0]

    t = int(lab[cookie_id])
    e = EV[EV.cookie_id == cookie_id]
    print(f"{cookie_id}   метка: {'БОТ' if t else 'человек'}   событий: {len(e)}")
    print(f"окно: {e.window_start_ts.iat[0].date()}   с {e.event_ts.min().time()} по {e.event_ts.max().time()}")

    cols = ["event_ts", "dt", "event_name", "platform_norm", "item_category",
            "search_page", "pointer_x", "pointer_y"]
    tl = e[cols].head(n).copy()
    tl["event_ts"] = tl["event_ts"].dt.strftime("%H:%M:%S")
    tl["dt"] = tl["dt"].round(1)
    display(tl.reset_index(drop=True))
    if len(e) > n:
        print(f"... показано {n} из {len(e)}")

    f = pd.DataFrame({"эта кука": X.loc[cookie_id, KEY],
                      "медиана людей": MED.loc[0, KEY],
                      "медиана ботов": MED.loc[1, KEY]}).round(3)
    display(f)
    return cookie_id

In [3]:
show(target=1, seed=1)      # случайный бот

ck_8da3acf08d1b9728   метка: БОТ   событий: 17
окно: 2026-04-07   с 18:17:14 по 20:23:11


,event_ts,dt,event_name,platform_norm,item_category,search_page,pointer_x,pointer_y
0,18:17:14,NaN,item_view,web,odezhda,NaN,NaN,NaN
1,18:17:33,19.0,item_view,web,detskie_tovary,NaN,NaN,NaN
2,18:18:09,36.0,photo_swipe,web,odezhda,NaN,NaN,NaN
3,18:19:06,57.0,search_results_view,desktop,odezhda,6.0,NaN,NaN
4,20:17:57,7131.0,search_results_view,desktop,detskie_tovary,1.0,NaN,NaN
5,20:18:12,15.0,search_results_view,web,odezhda,3.0,NaN,NaN
6,20:18:31,19.0,item_view,web,mebel,NaN,NaN,NaN
7,20:18:54,23.0,item_view,desktop,detskie_tovary,NaN,NaN,NaN
8,20:19:28,34.0,search_results_view,desktop,mebel,6.0,NaN,NaN
9,20:19:33,5.0,seller_page_view,web,odezhda,NaN,NaN,NaN


,эта кука,медиана людей,медиана ботов
n_events,17.000,12.000,20.000
dt_median,25.000,59.000,25.000
dt_cv,3.760,2.064,2.616
ptr_frac,0.000,0.000,0.000
max_search_page,6.000,4.000,7.000
n_cat_uniq,3.000,2.000,2.000
n_loc_uniq,12.000,5.000,8.000
item_pop_mean,0.556,0.750,1.227
photo_per_view,0.167,0.286,0.143
q_repeat_frac,0.375,0.000,0.333


'ck_8da3acf08d1b9728'

In [4]:
show(target=0, seed=1)      # случайный человек

ck_089765019e1b2292   метка: человек   событий: 10
окно: 2026-04-14   с 11:34:48 по 11:43:49


,event_ts,dt,event_name,platform_norm,item_category,search_page,pointer_x,pointer_y
0,11:34:48,NaN,search_results_view,web,bytovaya_tehnika,13.0,331.0,331.0
1,11:35:03,15.0,favorite_add,web,bytovaya_tehnika,NaN,1115.0,645.0
2,11:36:10,67.0,search_results_view,web,bytovaya_tehnika,4.0,1724.0,194.0
3,11:37:18,68.0,search_results_view,desktop,NaN,7.0,1314.0,365.0
4,11:38:36,78.0,photo_swipe,web,bytovaya_tehnika,NaN,1240.0,939.0
5,11:40:04,88.0,seller_page_view,web,bytovaya_tehnika,NaN,907.0,167.0
6,11:41:09,65.0,favorite_add,web,bytovaya_tehnika,NaN,1746.0,472.0
7,11:41:29,20.0,item_view,web,bytovaya_tehnika,NaN,530.0,570.0
8,11:42:57,88.0,item_view,web,bytovaya_tehnika,NaN,1522.0,274.0
9,11:43:49,52.0,seller_page_view,desktop,bytovaya_tehnika,NaN,701.0,337.0


,эта кука,медиана людей,медиана ботов
n_events,10.000,12.000,20.000
dt_median,67.000,59.000,25.000
dt_cv,0.445,2.064,2.616
ptr_frac,1.000,0.000,0.000
max_search_page,13.000,4.000,7.000
n_cat_uniq,1.000,2.000,2.000
n_loc_uniq,5.000,5.000,8.000
item_pop_mean,0.571,0.750,1.227
photo_per_view,0.500,0.286,0.143
q_repeat_frac,0.000,0.000,0.333


'ck_089765019e1b2292'

In [5]:
def compare(n=6, seed=0):
    """Свести несколько ботов и людей в одну таблицу по ключевым признакам."""
    b = pd.Series(lab[lab == 1].index).sample(n, random_state=seed)
    h = pd.Series(lab[lab == 0].index).sample(n, random_state=seed)
    t = X.loc[list(b) + list(h), KEY].round(2)
    t.insert(0, "кто", ["бот"] * n + ["человек"] * n)
    return t


display(compare())

,кто,n_events,dt_median,dt_cv,ptr_frac,max_search_page,n_cat_uniq,n_loc_uniq,item_pop_mean,photo_per_view,q_repeat_frac
ck_1ca07faa8ed764b2,бот,9,22.0,2.72,0.0,NaN,1,6,2.78,0.20,NaN
ck_73081e8253da24d2,бот,7,42.5,1.57,0.0,2.0,3,5,0.00,0.33,0.00
ck_9c9983d87f873af6,бот,41,15.5,2.80,0.0,14.0,2,12,1.48,0.12,0.38
ck_bf46eee06293ddb4,бот,46,9.0,0.41,0.0,22.0,3,17,0.00,0.11,0.50
ck_250c809ac1a2bdb8,бот,66,4.0,4.65,0.0,63.0,2,17,3.52,0.04,0.58
ck_cd4452be2e927b2b,бот,2,168.0,NaN,0.0,7.0,2,1,1.00,0.00,0.00
ck_31756087421c9296,человек,13,46.0,2.19,0.0,3.0,2,4,0.29,0.00,0.17
ck_0433129ac9f5c4cf,человек,40,76.0,2.37,0.0,10.0,3,10,0.53,0.24,0.12
ck_6705683b6e4c28d0,человек,13,96.5,2.01,0.0,2.0,2,3,0.25,0.20,0.20
ck_5630c741c0d36e7d,человек,8,67.0,2.37,1.0,3.0,3,5,1.00,2.00,0.00
